## 0. Setup — Local run

Run from the repository root (paths like `Part_2/...` assume it):

```bash
pip install -r Part_13/requirements.txt
jupyter notebook Part_13/Part_13_HybridGCNGAT_Local.ipynb
```

GPU is used automatically when available (`cuda`); CPU works, training just takes longer.

# Part 13: Hybrid GCN+GAT for MDM2 Classification

Trains a **hybrid graph network** from scratch on the 645 MDM2 compounds:
GCN layers first extract local structure, then a multi-head GAT layer refines
with learned neighbor attention, then mean+max pooling + classifier.

**Reference (hybrid design):** HDTI-IC50 — hybrid GCN+GAT model for p53-inhibitor
IC50 prediction, *J. Comput.-Aided Mol. Des.* 2025 (MAE 0.1, RMSE 0.19, R2 0.8,
beating single GCN/GAT). GCN: Kipf & Welling, ICLR 2017. GAT: Velickovic et al.,
ICLR 2018. Same 78-dim atom features and 645-compound protocol as Parts 9/10/12,
so results compare directly (RF 0.960 / GCN 0.952 / GIN 0.947).

**Pipeline:** SMILES → RDKit graphs → GCN(78→128)×2 → GAT(128→32×4 heads) → mean+max pool → 5-fold CV

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool, global_max_pool
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, roc_curve, auc, confusion_matrix
)
from sklearn.utils import shuffle
from rdkit import Chem
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 57
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Define Hybrid GCN+GAT Architecture

In [ ]:
class HybridGCNGAT(nn.Module):
    def __init__(self, num_node_features=78, gcn_hidden=128, gat_out=32, gat_heads=4,
                 num_classes=2, dropout=0.2):
        super().__init__()
        self.conv1 = GCNConv(num_node_features, gcn_hidden)
        self.conv2 = GCNConv(gcn_hidden, gcn_hidden)
        self.bn1 = nn.BatchNorm1d(gcn_hidden)
        self.bn2 = nn.BatchNorm1d(gcn_hidden)
        self.gat = GATConv(gcn_hidden, gat_out, heads=gat_heads, dropout=dropout)
        self.dropout = dropout
        self.classifier = nn.Sequential(
            nn.Linear(gat_out * gat_heads * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x, edge_index, batch):
        x = F.relu(self.bn1(self.conv1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.bn2(self.conv2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.gat(x, edge_index))
        x = torch.cat([global_mean_pool(x, batch), global_max_pool(x, batch)], dim=1)
        return self.classifier(x)

## 2. Load Data + Featurize

In [ ]:
df = pd.read_csv('Part_2/ro5_properties_filtered.csv')
df['y'] = df['bioactivity_class'].apply(lambda x: 1 if x == 'Active' else 0)
print(f"Dataset: {len(df)} compounds")
print(f"Class distribution:\n{df['y'].value_counts()}")

In [ ]:
ATOM_CHOICES = {
    'atomic_num': list(range(1, 101)),
    'degree': [0, 1, 2, 3, 4, 5],
    'formal_charge': [-2, -1, 0, 1, 2, 3],
    'num_hs': [0, 1, 2, 3, 4],
    'hybridization': [
        Chem.rdchem.HybridizationType.SP, Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3, Chem.rdchem.HybridizationType.SP3D,
        Chem.rdchem.HybridizationType.SP3D2
    ]
}

def one_hot(val, choices):
    enc = [0] * len(choices)
    if val in choices:
        enc[choices.index(val)] = 1
    return enc

def atom_features(atom):
    f = []
    f += one_hot(atom.GetAtomicNum(), ATOM_CHOICES['atomic_num'])
    f += one_hot(atom.GetTotalDegree(), ATOM_CHOICES['degree'])
    f += one_hot(atom.GetFormalCharge(), ATOM_CHOICES['formal_charge'])
    f += one_hot(atom.GetTotalNumHs(), ATOM_CHOICES['num_hs'])
    f += one_hot(atom.GetHybridization(), ATOM_CHOICES['hybridization'])
    f.append(int(atom.GetIsAromatic()))
    f.append(int(atom.IsInRing()))
    return (f + [0] * 78)[:78]

def mol_to_graph(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    node_features = [atom_features(a) for a in mol.GetAtoms()]
    edge_index = []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        edge_index.extend([[i, j], [j, i]])
    if not edge_index:
        edge_index = [[0, 0]]
    return Data(
        x=torch.tensor(node_features, dtype=torch.float),
        edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
        y=torch.tensor([label], dtype=torch.long)
    )

In [ ]:
graphs = []
for _, row in df.iterrows():
    g = mol_to_graph(row['canonical_smiles'], row['y'])
    if g is not None:
        graphs.append(g)

labels = np.array([g.y.item() for g in graphs])
print(f"Converted {len(graphs)}/{len(df)} molecules to graphs")
print(f"Example: {graphs[0].x.shape[0]} atoms, {graphs[0].edge_index.shape[1]} edges")

## 3. Train/Test Split

In [ ]:
train_idx, test_idx, y_train, y_test = train_test_split(
    np.arange(len(graphs)), labels, test_size=0.2, random_state=RANDOM_SEED, stratify=labels
)
train_graphs = [graphs[i] for i in train_idx]
test_graphs = [graphs[i] for i in test_idx]
train_loader = DataLoader(train_graphs, batch_size=64, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=64, shuffle=False)
print(f"Train: {len(train_graphs)}, Test: {len(test_graphs)}")

## 4. Training Functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    preds, probs, labels = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            probs.extend(F.softmax(out, dim=1)[:, 1].cpu().numpy())
            preds.extend(out.argmax(dim=1).cpu().numpy())
            labels.extend(batch.y.cpu().numpy())
    return np.array(preds), np.array(probs), np.array(labels)

## 5. Train Hybrid GCN+GAT

In [ ]:
model = HybridGCNGAT(num_node_features=78, gcn_hidden=128, gat_out=32,
                     gat_heads=4, num_classes=2, dropout=0.2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
EPOCHS = 100
train_losses, test_aucs = [], []

for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(model, train_loader, optimizer, criterion)
    train_losses.append(loss)
    if epoch % 10 == 0 or epoch == 1:
        preds, probs, true = evaluate(model, test_loader)
        auc_val = roc_auc_score(true, probs) if len(np.unique(true)) > 1 else 0
        test_aucs.append(auc_val)
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {loss:.4f} | Test AUC: {auc_val:.3f}")

print("\nTraining complete.")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(train_losses, 'b-', lw=1.5); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss'); ax1.grid(True, alpha=0.3)
ax2.plot(test_aucs, 'r-o', lw=1.5)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('ROC-AUC'); ax2.set_title('Test ROC-AUC'); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_curves_hybrid.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Test Set Evaluation

In [ ]:
test_pred, test_prob, test_true = evaluate(model, test_loader)
test_pred_binary = (test_prob > 0.5).astype(int)

test_results = pd.DataFrame([{
    'Model': 'Hybrid GCN+GAT',
    'Accuracy': accuracy_score(test_true, test_pred_binary),
    'Precision': precision_score(test_true, test_pred_binary, zero_division=0),
    'F1-score': f1_score(test_true, test_pred_binary, zero_division=0),
    'Sensitivity': recall_score(test_true, test_pred_binary, zero_division=0),
    'ROC-AUC': roc_auc_score(test_true, test_prob),
    'MCC': matthews_corrcoef(test_true, test_pred_binary)
}]).round(3)
test_results.to_csv('performance_hybrid_test.csv', index=False)
test_results

## 7. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(test_true, test_prob)
roc_auc_val = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', lw=2, label=f'Hybrid GCN+GAT (AUC = {roc_auc_val:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
plt.xlim([0, 1]); plt.ylim([0, 1.05])
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Hybrid GCN+GAT')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curve_hybrid.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Confusion Matrices

In [ ]:
cm = confusion_matrix(test_true, test_pred_binary)
class_labels = ['Inactive', 'Active']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True'); axes[0].set_title('Confusion Matrix')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, ax=axes[1])
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True'); axes[1].set_title('Normalized CM')
plt.tight_layout()
plt.savefig('confusion_matrix_hybrid.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Y-Randomization Test

In [ ]:
shuffled_results = []
for i in range(10):
    y_shuffled = shuffle(y_train, random_state=i)
    shuffled_graphs = [
        Data(x=g.x, edge_index=g.edge_index, y=torch.tensor([y_shuffled[j]], dtype=torch.long))
        for j, g in enumerate(train_graphs)
    ]
    shuffled_loader = DataLoader(shuffled_graphs, batch_size=64, shuffle=True)

    s_model = HybridGCNGAT(num_node_features=78, gcn_hidden=128, gat_out=32,
                           gat_heads=4, num_classes=2, dropout=0.2).to(device)
    s_opt = torch.optim.Adam(s_model.parameters(), lr=1e-3)
    for _ in range(100):
        train_epoch(s_model, shuffled_loader, s_opt, criterion)

    s_pred, s_prob, s_true = evaluate(s_model, test_loader)
    s_bin = (s_prob > 0.5).astype(int)
    shuffled_results.append({
        'Model': f'Shuffled Hybrid {i+1}',
        'Accuracy': round(accuracy_score(s_true, s_bin), 3),
        'F1-score': round(f1_score(s_true, s_bin, zero_division=0), 3),
        'ROC-AUC': round(roc_auc_score(s_true, s_prob), 3),
        'MCC': round(matthews_corrcoef(s_true, s_bin), 3)
    })
    print(f"Shuffle {i+1}/10 - ROC-AUC: {shuffled_results[-1]['ROC-AUC']:.3f}")

shuffled_df = pd.DataFrame(shuffled_results)
shuffled_df.to_csv('performance_hybrid_shuffled.csv', index=False)
shuffled_df

## 10. 5-Fold Cross-Validation

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
fold_metrics = []

all_labels = np.array([g.y.item() for g in graphs])

for fold, (tr_idx, val_idx) in enumerate(skf.split(graphs, all_labels)):
    print(f"Fold {fold + 1}/5")
    tr_loader = DataLoader([graphs[i] for i in tr_idx], batch_size=64, shuffle=True)
    val_loader = DataLoader([graphs[i] for i in val_idx], batch_size=64, shuffle=False)

    f_model = HybridGCNGAT(num_node_features=78, gcn_hidden=128, gat_out=32,
                           gat_heads=4, num_classes=2, dropout=0.2).to(device)
    f_opt = torch.optim.Adam(f_model.parameters(), lr=1e-3)
    for _ in range(100):
        train_epoch(f_model, tr_loader, f_opt, criterion)

    preds, probs, true = evaluate(f_model, val_loader)
    pred_bin = (probs > 0.5).astype(int)
    fold_metrics.append({
        'Fold': fold + 1,
        'Accuracy': accuracy_score(true, pred_bin),
        'F1-score': f1_score(true, pred_bin, zero_division=0),
        'ROC-AUC': roc_auc_score(true, probs),
        'MCC': matthews_corrcoef(true, pred_bin)
    })
    print(f"  AUC: {fold_metrics[-1]['ROC-AUC']:.3f} | F1: {fold_metrics[-1]['F1-score']:.3f}")

cv_results = pd.DataFrame(fold_metrics)
cv_results.loc['mean'] = cv_results.drop(columns='Fold').mean()
cv_results.loc['std'] = cv_results.drop(columns='Fold').std()
cv_results.to_csv('performance_hybrid_cv.csv', index=False)
cv_results

## 11. Save Model + Compare

In [ ]:
torch.save(model.state_dict(), 'hybrid_gcn_gat_mdm2.pth')
print("Model saved: Part_13/hybrid_gcn_gat_mdm2.pth")

In [ ]:
rows = []
try:
    rf = pd.read_csv('Part_6/performance_morgan_tuned.csv')
    r = rf[rf['Model'] == 'RandomForestClassifier'].iloc[0].to_dict()
    r['Model'] = 'RF (Morgan) - Part 6'; rows.append(r)
except Exception: pass
for fname, label in [('performance_gcn_test.csv', 'GCN (scratch) - Part 9'),
                     ('performance_pretrained_gnn_test.csv', 'GIN - Part 10'),
                     ('performance_attentivefp_test.csv', 'AttentiveFP - Part 12')]:
    try:
        t = pd.read_csv(fname)
        r = t.iloc[0].to_dict(); r['Model'] = label; rows.append(r)
    except Exception: pass
r = test_results.iloc[0].to_dict(); r['Model'] = 'Hybrid GCN+GAT - Part 13'; rows.append(r)

comparison = pd.DataFrame(rows).round(3)
comparison.to_csv('comparison_hybrid_vs_all.csv', index=False)

metrics_to_plot = ['Accuracy', 'Precision', 'F1-score', 'Sensitivity', 'ROC-AUC', 'MCC']
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_to_plot))
w = 0.35
rf_rows = comparison[comparison['Model'].str.contains('RF')]
hy_rows = comparison[comparison['Model'].str.contains('Hybrid')]
if len(rf_rows) and len(hy_rows):
    cols = [m for m in metrics_to_plot if m in rf_rows.columns and m in hy_rows.columns]
    ax.bar(x[:len(cols)] - w/2, rf_rows[cols].values.flatten(), w, label='Random Forest (Morgan)', color='steelblue')
    ax.bar(x[:len(cols)] + w/2, hy_rows[cols].values.flatten(), w, label='Hybrid GCN+GAT', color='darkorange')
    ax.legend()
    ax.set_xticks(x[:len(cols)]); ax.set_xticklabels(cols, rotation=45, ha='right')
ax.set_ylabel('Score'); ax.set_title('Random Forest vs Hybrid GCN+GAT')
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig('comparison_hybrid_vs_all.png', dpi=300, bbox_inches='tight')
plt.show()
comparison

In [ ]:
print("\n=== Part 13 Complete ===")
print("Outputs saved to Part_13/")